In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/model-dataset/Model_dataset.csv


In [2]:
import sys, subprocess

# Always install using the SAME python executable as the notebook kernel
PY = sys.executable

# (optional) remove wrong/old installs
subprocess.run([PY, "-m", "pip", "uninstall", "-y", "datasets", "pyarrow", "iterative-stratification"], check=False)

# Install a compatible stack (HF datasets + pyarrow) + iterative stratification
subprocess.check_call([PY, "-m", "pip", "install", "-U",
                       "pyarrow==15.0.2",
                       "datasets==2.19.2",
                       "transformers==4.47.1",
                       "accelerate>=0.34.0",
                       "evaluate>=0.4.2",
                       "iterative-stratification==0.1.9"])
print("✅ installs done for:", PY)


Found existing installation: datasets 4.4.1
Uninstalling datasets-4.4.1:
  Successfully uninstalled datasets-4.4.1
Found existing installation: pyarrow 19.0.1
Uninstalling pyarrow-19.0.1:
  Successfully uninstalled pyarrow-19.0.1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.5 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.3/38.3 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.1/542.1 kB 32.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 132.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 380.9/380.9 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.12.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
s3fs 2025.3.0 requires fsspec==2025.3.0.*, but you have fsspec 2024.3.1 which is incompatible.
cesium 0.12.4 requires numpy<3.0,>=2.0, but you have numpy 1.26.4 which is incompatible.
bigframes 2.12.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
libcugraph-cu12 25.6.0 requires libraft-cu12==25.6.*, but you have libraft-cu12 25.2.0 which is incompatible.
gradio 5.38.1 requires pydantic<2.12,>=2.0, but you have pydantic 2.12.4 which is incompatible.
cudf-polars-cu12 25.6.0 requires pylibcudf-cu12==25.6.*, but you have pylibcudf-cu12 25.2.2 which is incompatible.
pylibcugraph-cu12 25.6.0 requires pylibraft-cu12==25.6.*, but you have pylibraft-cu12 25.2.0 which is incompatible.
pylibcugraph-cu1

✅ installs done for: /usr/bin/python3


In [3]:
import os, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# keep these before transformers
os.environ["DISABLE_TRANSFORMERS_INTEGRATION_DATASETS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

from sklearn.metrics import f1_score, precision_score, recall_score

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    set_seed
)

# ✅ version check
import datasets, pyarrow, transformers
print("✅ IMPORTS OK")
print("datasets:", datasets.__version__)
print("pyarrow:", pyarrow.__version__)
print("transformers:", transformers.__version__)


<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.Tensor size changed, may indicate binary incompatibility. Expected 64 from C header, got 80 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.ChunkedArray size changed, may indicate binary incompatibility. Expected 64 from C header, got 72 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib._Tabular size changed, may indicate binary incompatibility. Expected 24 from C header, got 32 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.Table size changed, may indicate binary incompatibility. Expected 56 from C header, got 64 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.Tensor size changed, may indicate binary incompatibility. Expected 64 from C header, got 80 from PyObject
<frozen importlib._bootstrap>:241: RuntimeWarning: pyarrow.lib.ChunkedArray size changed, may indicate binary incompatibility. Expected 64 from C he

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

✅ IMPORTS OK
datasets: 2.19.2
pyarrow: 19.0.1
transformers: 4.47.1


In [4]:

# ---------------------------
# ✅ Config
# ---------------------------
SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "xlm-roberta-base"          # P100 friendly; strong for Banglish
MAX_LEN    = 128                         # speed; try 160 if still fast
EPOCHS     = 10
LR         = 2e-5
WD         = 0.01
WARMUP     = 0.10
OUTDIR     = "/kaggle/working/xlmr-attn-bce-banglish"

In [5]:
# ---------------------------
# ✅ Load Data
# ---------------------------
# 🔥 Change path to your dataset file
DATA_PATH = "/kaggle/input/model-dataset/Model_dataset.csv"

df = pd.read_csv(DATA_PATH)

# Clean column names
if isinstance(df, pd.DataFrame):
    df.columns = df.columns.str.strip()

TEXT_COL = "Data"
LABEL_COLS = ["Love", "Joy", "Anger", "Surprise", "Sadness", "Fear", "Hate"]

# Basic checks
missing = [c for c in [TEXT_COL] + LABEL_COLS if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in CSV: {missing}\nFound: {df.columns.tolist()}")

df = df[[TEXT_COL] + LABEL_COLS].copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).fillna("")
for c in LABEL_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# Remove empty texts (optional)
df = df[df[TEXT_COL].str.strip().astype(bool)].reset_index(drop=True)

print("✅ Samples:", len(df))
print("✅ Labels:", LABEL_COLS)
print(df.head(2))

✅ Samples: 27809
✅ Labels: ['Love', 'Joy', 'Anger', 'Surprise', 'Sadness', 'Fear', 'Hate']
                                                Data  Love  Joy  Anger  \
0  atae best story amar darun lagacha wow smiling...     0    0      0   
1                 onek valo hoto jodi phone ta patam     0    0      0   

   Surprise  Sadness  Fear  Hate  
0         1        0     0     0  
1         0        1     0     0  


In [6]:
# ---------------------------
# ✅ Split (70/15/15) — NO iterstrat needed
# ---------------------------
from sklearn.model_selection import train_test_split

# 1) 70 train, 30 temp
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=SEED,
    shuffle=True
)

# 2) temp -> 15 val, 15 test  (half of 30%)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    shuffle=True
)

# reset index (keeps rest of notebook unchanged)
train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f"✅ Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")


✅ Train: 19466 | Val: 4171 | Test: 4172


In [7]:
# ---------------------------
# ✅ Tokenizer + Datasets
# ---------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def to_hf_dataset(pdf: pd.DataFrame):
    return Dataset.from_pandas(pdf, preserve_index=False)

train_ds = to_hf_dataset(train_df)
val_ds   = to_hf_dataset(val_df)
test_ds  = to_hf_dataset(test_df)

def tokenize_fn(batch):
    return tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LEN
    )

train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=[])
val_ds   = val_ds.map(tokenize_fn, batched=True, remove_columns=[])
test_ds  = test_ds.map(tokenize_fn, batched=True, remove_columns=[])

# Trainer will look for "labels"
def add_labels(batch):
    labels = np.stack([batch[c] for c in LABEL_COLS], axis=1).astype(np.float32)
    batch["labels"] = labels
    return batch

train_ds = train_ds.map(add_labels, batched=True)
val_ds   = val_ds.map(add_labels, batched=True)
test_ds  = test_ds.map(add_labels, batched=True)

cols_to_keep = ["input_ids", "attention_mask", "labels"]
train_ds.set_format(type="torch", columns=cols_to_keep)
val_ds.set_format(type="torch", columns=cols_to_keep)
test_ds.set_format(type="torch", columns=cols_to_keep)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

Map:   0%|          | 0/19466 [00:00<?, ? examples/s]

Map:   0%|          | 0/4171 [00:00<?, ? examples/s]

Map:   0%|          | 0/4172 [00:00<?, ? examples/s]

Map:   0%|          | 0/19466 [00:00<?, ? examples/s]

Map:   0%|          | 0/4171 [00:00<?, ? examples/s]

Map:   0%|          | 0/4172 [00:00<?, ? examples/s]

In [8]:
# ---------------------------
# ✅ POS_WEIGHT (imbalance fix)
# ---------------------------
train_y = train_df[LABEL_COLS].values.astype(np.float32)
pos = train_y.sum(axis=0)
neg = (train_y.shape[0] - pos)
pos_weight = (neg / (pos + 1e-6))
pos_weight_t = torch.tensor(pos_weight, dtype=torch.float32)
print("✅ pos_weight:", dict(zip(LABEL_COLS, pos_weight.round(2))))

✅ pos_weight: {'Love': 5.34, 'Joy': 5.41, 'Anger': 5.42, 'Surprise': 5.49, 'Sadness': 5.42, 'Fear': 5.45, 'Hate': 5.38}


In [9]:
# ---------------------------
# ✅ XLM-R + Attention Pooling Model
# ---------------------------
from transformers import AutoModel, AutoConfig
import torch
import torch.nn as nn

class XLMRAttentionForMultilabel(nn.Module):
    def __init__(self, model_name, num_labels, pos_weight):
        super().__init__()
        self.config = AutoConfig.from_pretrained(model_name)
        self.backbone = AutoModel.from_pretrained(model_name, config=self.config)

        self.attn = nn.Linear(self.config.hidden_size, 1)
        self.classifier = nn.Linear(self.config.hidden_size, num_labels)
        self.dropout = nn.Dropout(0.2)

        self.register_buffer("pos_weight", pos_weight)

    def gradient_checkpointing_enable(self, **kwargs):
        self.backbone.gradient_checkpointing_enable(**kwargs)

    def gradient_checkpointing_disable(self):
        self.backbone.gradient_checkpointing_disable()

    def forward(self, input_ids=None, attention_mask=None, labels=None):
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask)
        x = outputs.last_hidden_state

        attn_scores = self.attn(x).squeeze(-1)
        mask_val = torch.finfo(attn_scores.dtype).min
        attn_scores = attn_scores.masked_fill(attention_mask == 0, mask_val)

        attn_weights = torch.softmax(attn_scores, dim=1).unsqueeze(-1)
        pooled = torch.sum(x * attn_weights, dim=1)

        pooled = self.dropout(pooled)
        logits = self.classifier(pooled)

        loss = None
        if labels is not None:
            loss = nn.BCEWithLogitsLoss(pos_weight=self.pos_weight)(logits, labels)

        return {"loss": loss, "logits": logits}



model = XLMRAttentionForMultilabel(
    model_name=MODEL_NAME,
    num_labels=len(LABEL_COLS),
    pos_weight=pos_weight_t
)

# Memory/speed tricks
model.backbone.gradient_checkpointing_enable()

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

In [10]:
# ---------------------------
# ✅ Metrics + Threshold Tuning
# ---------------------------
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = sigmoid(logits)

    # default threshold 0.5 during training
    preds = (probs >= 0.5).astype(int)

    micro_f1  = f1_score(labels, preds, average="micro", zero_division=0)
    macro_f1  = f1_score(labels, preds, average="macro", zero_division=0)
    micro_p   = precision_score(labels, preds, average="micro", zero_division=0)
    micro_r   = recall_score(labels, preds, average="micro", zero_division=0)

    # "subset accuracy" = exact match of all labels
    subset_acc = (preds == labels).all(axis=1).mean()

    # label-wise accuracy average
    label_acc = (preds == labels).mean()

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "micro_precision": micro_p,
        "micro_recall": micro_r,
        "subset_accuracy": subset_acc,
        "label_accuracy": label_acc,
    }

def find_best_thresholds(model, trainer, val_ds, label_cols):
    # Predict on val
    pred = trainer.predict(val_ds)
    logits = pred.predictions
    y_true = pred.label_ids
    probs = sigmoid(logits)

    best_thr = []
    for i in range(y_true.shape[1]):
        best_t, best_f1 = 0.5, -1
        # quick grid (fast)
        for t in np.linspace(0.15, 0.85, 29):
            y_pred = (probs[:, i] >= t).astype(int)
            f1 = f1_score(y_true[:, i], y_pred, zero_division=0)
            if f1 > best_f1:
                best_f1 = f1
                best_t = float(t)
        best_thr.append(best_t)

    best_thr = np.array(best_thr, dtype=np.float32)
    print("✅ Best thresholds:", dict(zip(label_cols, np.round(best_thr, 3))))
    return best_thr

def evaluate_with_thresholds(trainer, ds, thresholds):
    pred = trainer.predict(ds)
    logits = pred.predictions
    y_true = pred.label_ids
    probs = sigmoid(logits)

    y_pred = (probs >= thresholds.reshape(1, -1)).astype(int)

    micro_f1 = f1_score(y_true, y_pred, average="micro", zero_division=0)
    macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
    micro_p  = precision_score(y_true, y_pred, average="micro", zero_division=0)
    micro_r  = recall_score(y_true, y_pred, average="micro", zero_division=0)
    subset_acc = (y_pred == y_true).all(axis=1).mean()
    label_acc  = (y_pred == y_true).mean()

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "micro_precision": micro_p,
        "micro_recall": micro_r,
        "subset_accuracy": subset_acc,
        "label_accuracy": label_acc,
    }


In [11]:
# ---------------------------
# ✅ TrainingArguments (P100 speed target)
# ---------------------------
training_args = TrainingArguments(
    output_dir=OUTDIR,
    seed=SEED,

    num_train_epochs=EPOCHS,
    learning_rate=LR,
    weight_decay=WD,
    warmup_ratio=WARMUP,
    lr_scheduler_type="cosine",

    per_device_train_batch_size=16,       # P100 usually OK @ MAX_LEN 128
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,        # effective batch = 32

    fp16=True,                            # speed + memory
    gradient_checkpointing=True,

    logging_steps=200,
    eval_strategy="steps",
    eval_steps=400,
    save_strategy="steps",
    save_steps=400,
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="micro_f1",
    greater_is_better=True,

    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)


In [12]:
# ---------------------------
# ✅ Train
# ---------------------------
trainer.train()
print("✅ Best checkpoint:", trainer.state.best_model_checkpoint)

Step,Training Loss,Validation Loss,Micro F1,Macro F1,Micro Precision,Micro Recall,Subset Accuracy,Label Accuracy
400,2.049300,0.975999,0.388554,0.421288,0.261846,0.752866,0.039559,0.631880
800,1.696900,0.837524,0.475765,0.499983,0.343298,0.774691,0.056341,0.734767
1200,1.532000,0.742257,0.528910,0.549848,0.393441,0.806658,0.153440,0.776758
1600,1.375400,0.699159,0.566870,0.572789,0.433559,0.818563,0.199233,0.805665
2000,1.222700,0.685853,0.589450,0.594392,0.462737,0.811728,0.270199,0.824331
2400,1.137700,0.670321,0.599641,0.604936,0.475802,0.810626,0.295373,0.831832
2800,1.032200,0.683585,0.610286,0.616518,0.492618,0.801808,0.333014,0.840908
3200,0.903600,0.667194,0.621115,0.629159,0.503423,0.810626,0.343323,0.846354
3600,0.882200,0.662935,0.626967,0.635925,0.510459,0.812390,0.354351,0.849813
4000,0.766800,0.681027,0.634926,0.641041,0.523946,0.805556,0.377368,0.856081


✅ Best checkpoint: /kaggle/working/xlmr-attn-bce-banglish/checkpoint-5200


In [13]:

# ---------------------------
# ✅ Threshold tuning on Val (big F1 boost)
# ---------------------------
best_thresholds = find_best_thresholds(model, trainer, val_ds, LABEL_COLS)

print("\n✅ VAL (threshold-tuned) metrics:")
val_metrics = evaluate_with_thresholds(trainer, val_ds, best_thresholds)
for k,v in val_metrics.items():
    print(f"{k}: {v:.4f}")

print("\n✅ TEST (threshold-tuned) metrics:")
test_metrics = evaluate_with_thresholds(trainer, test_ds, best_thresholds)
for k,v in test_metrics.items():
    print(f"{k}: {v:.4f}")

✅ Best thresholds: {'Love': 0.675, 'Joy': 0.525, 'Anger': 0.775, 'Surprise': 0.725, 'Sadness': 0.75, 'Fear': 0.825, 'Hate': 0.5}

✅ VAL (threshold-tuned) metrics:


micro_f1: 0.6653
macro_f1: 0.6680
micro_precision: 0.6103
micro_recall: 0.7310
subset_accuracy: 0.5121
label_accuracy: 0.8857

✅ TEST (threshold-tuned) metrics:


micro_f1: 0.6661
macro_f1: 0.6688
micro_precision: 0.6131
micro_recall: 0.7290
subset_accuracy: 0.5194
label_accuracy: 0.8864


In [14]:
# ---------------------------
# ✅ Save model + tokenizer + thresholds
# ---------------------------
SAVE_DIR = "/kaggle/working/final_xlmr_attn_model"
os.makedirs(SAVE_DIR, exist_ok=True)

# Save Trainer model
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)

# Save thresholds
np.save(os.path.join(SAVE_DIR, "best_thresholds.npy"), best_thresholds)

print("\n✅ Saved to:", SAVE_DIR)
print("✅ Files:", os.listdir(SAVE_DIR))


✅ Saved to: /kaggle/working/final_xlmr_attn_model
✅ Files: ['tokenizer_config.json', 'tokenizer.json', 'training_args.bin', 'best_thresholds.npy', 'model.safetensors', 'sentencepiece.bpe.model', 'special_tokens_map.json']


In [16]:
# ===========================
# ✅ Overall (Label-wise) Accuracy — Subset accuracy ছাড়াই (FIXED)
# ===========================
import numpy as np

def _sigmoid(x):
    return 1 / (1 + np.exp(-x))

# 1) Get logits + true labels from Trainer
pred_out = trainer.predict(test_ds)
logits = pred_out.predictions
y_true = pred_out.label_ids.astype(int)

# 2) Apply tuned thresholds (per-label)
probs = _sigmoid(logits)

# ✅ FIX: best_thresholds can be dict OR list/np.array
if isinstance(best_thresholds, dict):
    thr_arr = np.array([float(best_thresholds.get(l, 0.5)) for l in LABEL_COLS], dtype=float)
else:
    bt = np.asarray(best_thresholds, dtype=float).reshape(-1)
    if bt.size < len(LABEL_COLS):
        raise ValueError(f"best_thresholds has {bt.size} values but needs {len(LABEL_COLS)}")
    thr_arr = bt[:len(LABEL_COLS)]

y_pred = (probs >= thr_arr).astype(int)

# ✅ Overall accuracy (label-wise / micro across all labels)
overall_label_accuracy = (y_pred == y_true).mean()

# ✅ Macro label accuracy (average of per-label accuracies)
per_label_acc = (y_pred == y_true).mean(axis=0)
macro_label_accuracy = float(per_label_acc.mean())

print("\n✅ Overall (label-wise) accuracy:", round(float(overall_label_accuracy), 6))
print("✅ Macro label accuracy:", round(macro_label_accuracy, 6))

print("\n✅ Per-label accuracy:")
for name, acc in zip(LABEL_COLS, per_label_acc):
    print(f"  {name:>9s}: {acc:.6f}")



✅ Overall (label-wise) accuracy: 0.886385
✅ Macro label accuracy: 0.886385

✅ Per-label accuracy:
       Love: 0.898610
        Joy: 0.857143
      Anger: 0.845158
   Surprise: 0.924976
    Sadness: 0.867929
       Fear: 0.937440
       Hate: 0.873442
